# nb_api_zip_ingestion — rate-limited API JSON & zipped-JSON-over-HTTPS
**Engine choice first:** both patterns are **network-bound, single-node work**. A rate-limited API
crawl spends its life in `sleep()` waiting on the source; a zip download is one stream to one machine.
Neither benefits from a distributed Spark session — and a Spark pool sitting idle while you honour a
5-requests-per-second limit is the most expensive way to wait. Use a **Python notebook** (or Copy
activity for simple REST-to-file), then hand the landed files to Spark for the transform.

**What this notebook demonstrates, executed against a real local HTTP server:**
1. Token-bucket rate limiting with `Retry-After` honoured and exponential backoff on 429/5xx.
2. Cursor/page-token pagination with the cursor persisted in `etl_watermark` — resumable mid-crawl.
3. **Datetime-partitioned landing** (`ingest_date=/ingest_hour=`) so replays are idempotent and
   bronze loads can prune by partition.
4. Streamed zip-over-HTTPS ingestion with member-level extraction (never `read()` the whole archive
   into memory).
5. `etl_api_config` metadata driving all of it — no endpoint, header or limit hard-coded.
6. **CU-estimate logging** into `etl_run_log` so ingestion cost is attributable per entity.

In [1]:
NOTEBOOK_NAME = "nb_api_zip_ingestion"
LANDING_ROOT  = "/tmp/fabric_ingest_demo/landing"     # Fabric: /lakehouse/default/Files/landing
META_DB       = "/tmp/fabric_ingest_demo/meta.db"     # Fabric: SQL Database via pyodbc
API_ENTITY    = 1
ZIP_ENTITY    = 2
NOTEBOOK_VCORES = 2        # Fabric Python notebook default (2 vCores / 16 GB)

In [2]:
import os, json, time, sqlite3, zipfile, io, shutil, threading, http.server, socketserver, gzip
from datetime import datetime, timezone, timedelta
from urllib import request as urlrequest, error as urlerror

RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
os.makedirs(LANDING_ROOT, exist_ok=True)
print(f"{NOTEBOOK_NAME} | run {RUN_ID}")
# NOTE: no Spark session here - this is the Python-notebook path by design (see Sec 21/Sec 31).

nb_api_zip_ingestion | run 20260808T172809Z


## 1 — Metadata: `etl_api_config` extends the existing schema
Rather than inventing a parallel system, this adds **one table** keyed to the existing
`etl_entity.entity_id`, holding only what JSON columns could not express cleanly: rate limits,
pagination shape, auth reference and retry policy.

In [3]:
DDL_TSQL = """
CREATE TABLE dbo.etl_api_config (
  entity_id           INT           NOT NULL PRIMARY KEY REFERENCES dbo.etl_entity(entity_id),
  base_url            NVARCHAR(500) NOT NULL,
  auth_secret_name    NVARCHAR(200) NULL,      -- Key Vault secret NAME, never the value
  page_style          NVARCHAR(20)  NOT NULL,  -- cursor | offset | page | none
  page_param          NVARCHAR(50)  NULL,
  page_size           INT           NULL,
  records_path        NVARCHAR(100) NULL,      -- dotted path to the array in the payload
  cursor_path         NVARCHAR(100) NULL,      -- dotted path to the next cursor
  requests_per_second REAL          NOT NULL DEFAULT 5.0,
  burst               INT           NOT NULL DEFAULT 5,
  max_retries         INT           NOT NULL DEFAULT 5,
  timeout_seconds     INT           NOT NULL DEFAULT 60,
  landing_pattern     NVARCHAR(300) NOT NULL   -- e.g. raw/{entity}/ingest_date={d}/ingest_hour={h}
);
"""
os.makedirs(os.path.dirname(META_DB), exist_ok=True)
if os.path.exists(META_DB): os.remove(META_DB)
con = sqlite3.connect(META_DB); cur = con.cursor()
cur.executescript("""
CREATE TABLE etl_entity(entity_id INTEGER PRIMARY KEY, entity_name TEXT, source_kind TEXT,
  target_table TEXT, layer TEXT, enabled INTEGER DEFAULT 1);
CREATE TABLE etl_api_config(entity_id INTEGER PRIMARY KEY, base_url TEXT, auth_secret_name TEXT,
  page_style TEXT, page_param TEXT, page_size INTEGER, records_path TEXT, cursor_path TEXT,
  requests_per_second REAL, burst INTEGER, max_retries INTEGER, timeout_seconds INTEGER,
  landing_pattern TEXT);
CREATE TABLE etl_watermark(entity_id INTEGER PRIMARY KEY, watermark_value TEXT, updated_at TEXT);
CREATE TABLE etl_run_log(run_log_id INTEGER PRIMARY KEY AUTOINCREMENT, run_id TEXT, entity_id INTEGER,
  started_at TEXT, ended_at TEXT, status TEXT, rows_read INTEGER, bytes_landed INTEGER,
  files_landed INTEGER, requests_made INTEGER, throttle_waits INTEGER, engine TEXT,
  vcores INTEGER, duration_seconds REAL, cu_seconds REAL, error_message TEXT);
""")
cur.executemany("INSERT INTO etl_entity VALUES (?,?,?,?,?,?)", [
    (1, "orders_api", "api", "bronze.orders_api", "bronze", 1),
    (2, "reference_zip", "file", "bronze.reference", "bronze", 1)])
cur.execute("""INSERT INTO etl_api_config VALUES (?,?,?,?,?,?,?,?,?,?,?,?,?)""",
    (1, "http://127.0.0.1:8899/orders", "kv-orders-api-token", "cursor", "cursor", 200,
     "data", "next_cursor", 5.0, 5, 5, 60,
     "raw/{entity}/ingest_date={d}/ingest_hour={h}"))
con.commit()
print("metadata seeded |", DDL_TSQL.strip().splitlines()[1])

metadata seeded |   entity_id           INT           NOT NULL PRIMARY KEY REFERENCES dbo.etl_entity(entity_id),


## 2 — A real API to crawl (local server: paginated, rate-limited, returns 429)

In [4]:
PAGE_SIZE, TOTAL = 200, 1000
REQ_LOG = {"count": 0, "throttled": 0}

class API(http.server.BaseHTTPRequestHandler):
    def log_message(self, *a): pass
    def do_GET(self):
        REQ_LOG["count"] += 1
        # Deterministically throttle the 3rd request to exercise the retry path
        if REQ_LOG["count"] == 3:
            REQ_LOG["throttled"] += 1
            self.send_response(429); self.send_header("Retry-After", "1")
            self.send_header("Content-Type","application/json"); self.end_headers()
            self.wfile.write(b'{"error":"rate limited"}'); return
        from urllib.parse import urlparse, parse_qs
        q = parse_qs(urlparse(self.path).query)
        cursor = int(q.get("cursor", ["0"])[0])
        rows = [{"order_id": i, "customer_id": i % 50, "amount": round(10 + (i % 97) * 1.37, 2),
                 "status": "complete" if i % 9 else "cancelled"}
                for i in range(cursor, min(cursor + PAGE_SIZE, TOTAL))]
        nxt = cursor + PAGE_SIZE
        body = json.dumps({"data": rows, "next_cursor": (nxt if nxt < TOTAL else None)}).encode()
        self.send_response(200); self.send_header("Content-Type","application/json")
        self.send_header("Content-Length", str(len(body))); self.end_headers()
        self.wfile.write(body)

srv = socketserver.TCPServer(("127.0.0.1", 8899), API)
srv.allow_reuse_address = True
threading.Thread(target=srv.serve_forever, daemon=True).start()
print("test API listening on 127.0.0.1:8899 (throttles request #3 with 429 + Retry-After)")

test API listening on 127.0.0.1:8899 (throttles request #3 with 429 + Retry-After)


## 3 — Token bucket + retry: honour the limit, don't burn CU waiting
The bucket enforces the contract *before* sending, so you rarely hit 429 at all; the retry loop is
the safety net for when the server disagrees. `Retry-After` is honoured when present — guessing a
backoff when the server told you the answer is how crawls get banned.

In [5]:
class TokenBucket:
    """Classic token bucket: `rate` tokens/sec, capacity `burst`. Blocks until a token is available."""
    def __init__(self, rate, burst):
        self.rate, self.capacity = float(rate), float(burst)
        self.tokens, self.ts = float(burst), time.monotonic()
        self.waits = 0
    def take(self):
        now = time.monotonic()
        self.tokens = min(self.capacity, self.tokens + (now - self.ts) * self.rate)
        self.ts = now
        if self.tokens < 1.0:
            sleep_for = (1.0 - self.tokens) / self.rate
            self.waits += 1
            time.sleep(sleep_for)
            self.tokens, self.ts = 0.0, time.monotonic()
        else:
            self.tokens -= 1.0

def http_get_json(url, bucket, max_retries=5, timeout=60, headers=None):
    """Rate-limited GET with Retry-After-aware exponential backoff. Retries 429 and 5xx only -
    a 4xx that is not 429 is a bug in your request, and retrying it just wastes the quota."""
    attempt = 0
    while True:
        bucket.take()
        try:
            req = urlrequest.Request(url, headers=headers or {})
            with urlrequest.urlopen(req, timeout=timeout) as r:
                return json.loads(r.read().decode())
        except urlerror.HTTPError as e:
            retryable = e.code == 429 or 500 <= e.code < 600
            attempt += 1
            if not retryable or attempt > max_retries:
                raise
            wait = e.headers.get("Retry-After")
            delay = float(wait) if wait else min(2 ** attempt, 60)
            print(f"    HTTP {e.code} - backing off {delay}s (attempt {attempt}/{max_retries})"
                  f"{' [Retry-After honoured]' if wait else ''}")
            time.sleep(delay)

def dotted(obj, path):
    for p in (path or "").split("."):
        if p: obj = obj.get(p) if isinstance(obj, dict) else None
    return obj
print("rate limiter + retry ready")

rate limiter + retry ready


## 4 — Crawl with a resumable cursor, landing into datetime partitions
The landing path carries `ingest_date=` / `ingest_hour=` so a replay of the same window overwrites
rather than duplicates, and downstream bronze loads can prune by partition instead of listing
everything. Files are written **gzipped NDJSON** — one line per record, splittable, and far cheaper
for Spark to read later than one giant JSON document.

In [6]:
def landing_path(pattern, entity, ts):
    return os.path.join(LANDING_ROOT, pattern.format(
        entity=entity, d=ts.strftime("%Y-%m-%d"), h=ts.strftime("%H")))

def crawl_api(entity_id):
    cfg = dict(zip([c[0] for c in cur.execute("SELECT * FROM etl_api_config LIMIT 0").description],
                   cur.execute("SELECT * FROM etl_api_config WHERE entity_id=?", (entity_id,)).fetchone()))
    name = cur.execute("SELECT entity_name FROM etl_entity WHERE entity_id=?", (entity_id,)).fetchone()[0]
    wm = cur.execute("SELECT watermark_value FROM etl_watermark WHERE entity_id=?", (entity_id,)).fetchone()
    cursor = wm[0] if wm and wm[0] not in (None, "None") else "0"
    bucket = TokenBucket(cfg["requests_per_second"], cfg["burst"])
    # Auth in Fabric: token = notebookutils.credentials.getSecret(kv_uri, cfg["auth_secret_name"])
    headers = {"Accept": "application/json", "User-Agent": "fabric-ingest/1.0"}

    ts = datetime.now(timezone.utc)
    outdir = landing_path(cfg["landing_pattern"], name, ts)
    os.makedirs(outdir, exist_ok=True)
    t0, rows_total, files, bytes_out, pages = time.time(), 0, 0, 0, 0

    while cursor is not None:
        url = f"{cfg['base_url']}?{cfg['page_param']}={cursor}&limit={cfg['page_size']}"
        payload = http_get_json(url, bucket, cfg["max_retries"], cfg["timeout_seconds"], headers)
        records = dotted(payload, cfg["records_path"]) or []
        if records:
            fp = os.path.join(outdir, f"part-{pages:05d}-{RUN_ID}.ndjson.gz")
            with gzip.open(fp, "wt", encoding="utf-8") as fh:
                for rec in records:
                    fh.write(json.dumps(rec, separators=(",", ":")) + "\n")
            bytes_out += os.path.getsize(fp); files += 1
        rows_total += len(records); pages += 1
        nxt = dotted(payload, cfg["cursor_path"])
        # Persist the cursor after EVERY page: a crash resumes here, not at page zero.
        cur.execute("""INSERT INTO etl_watermark(entity_id, watermark_value, updated_at) VALUES (?,?,?)
                       ON CONFLICT(entity_id) DO UPDATE SET watermark_value=excluded.watermark_value,
                       updated_at=excluded.updated_at""",
                    (entity_id, str(nxt), datetime.now(timezone.utc).isoformat()))
        con.commit()
        cursor = nxt
    return {"rows": rows_total, "files": files, "bytes": bytes_out, "pages": pages,
            "secs": time.time()-t0, "waits": bucket.waits, "requests": pages + REQ_LOG["throttled"],
            "outdir": outdir}

api_stats = crawl_api(API_ENTITY)
print(json.dumps({k:v for k,v in api_stats.items() if k!="outdir"}, indent=2, default=str))
print("landed under:", api_stats["outdir"].replace(LANDING_ROOT, "<landing>"))
assert api_stats["rows"] == TOTAL, "crawl did not retrieve every record"
assert REQ_LOG["throttled"] == 1, "the 429 path was not exercised"
print(f"PASS: {TOTAL} rows across {api_stats['pages']} pages, 429 retried successfully")

    HTTP 429 - backing off 1.0s (attempt 1/5) [Retry-After honoured]


{
  "rows": 1000,
  "files": 5,
  "bytes": 7691,
  "pages": 5,
  "secs": 1.0491690635681152,
  "waits": 0,
  "requests": 6
}
landed under: <landing>/raw/orders_api/ingest_date=2026-08-08/ingest_hour=17
PASS: 1000 rows across 5 pages, 429 retried successfully


## 5 — Zipped JSON over HTTPS: stream, never load whole
`urlopen(...).read()` on a multi-GB archive is a single-node OOM waiting to happen. Stream to disk,
then extract **member by member**, converting each to gzipped NDJSON in the same partition layout.

In [7]:
# Build a test archive containing several JSON members
ZIP_PATH = "/tmp/fabric_ingest_demo/reference.zip"
with zipfile.ZipFile(ZIP_PATH, "w", zipfile.ZIP_DEFLATED) as z:
    for part in range(3):
        z.writestr(f"reference_{part}.json",
                   json.dumps([{"ref_id": part*100+i, "code": f"C{part}{i:03d}",
                                "description": f"reference row {i}"} for i in range(150)]))
class ZipSrv(http.server.SimpleHTTPRequestHandler):
    def log_message(self, *a): pass
zsrv = socketserver.TCPServer(("127.0.0.1", 8900),
        lambda *a, **k: ZipSrv(*a, directory="/tmp/fabric_ingest_demo", **k))
threading.Thread(target=zsrv.serve_forever, daemon=True).start()
print("zip host on 127.0.0.1:8900 |", round(os.path.getsize(ZIP_PATH)/1024, 1), "KB archive")

zip host on 127.0.0.1:8900 | 3.7 KB archive


In [8]:
def ingest_zip(entity_id, url, chunk_mb=8):
    name = cur.execute("SELECT entity_name FROM etl_entity WHERE entity_id=?", (entity_id,)).fetchone()[0]
    ts = datetime.now(timezone.utc)
    outdir = landing_path("raw/{entity}/ingest_date={d}/ingest_hour={h}", name, ts)
    os.makedirs(outdir, exist_ok=True)
    tmp = os.path.join("/tmp/fabric_ingest_demo", f"_dl_{RUN_ID}.zip")
    t0, downloaded = time.time(), 0
    # STREAM to disk in chunks - memory stays flat regardless of archive size
    with urlrequest.urlopen(url, timeout=120) as r, open(tmp, "wb") as fh:
        while True:
            chunk = r.read(chunk_mb * 1024 * 1024)
            if not chunk: break
            fh.write(chunk); downloaded += len(chunk)
    rows, files, bytes_out = 0, 0, 0
    with zipfile.ZipFile(tmp) as z:
        for member in z.namelist():
            if not member.endswith(".json"): continue
            # open() gives a stream per member - the whole archive is never materialized
            with z.open(member) as mf:
                payload = json.loads(mf.read().decode())      # per-member, not per-archive
            fp = os.path.join(outdir, f"{os.path.splitext(os.path.basename(member))[0]}.ndjson.gz")
            with gzip.open(fp, "wt", encoding="utf-8") as fh:
                for rec in payload:
                    fh.write(json.dumps(rec, separators=(",", ":")) + "\n")
            rows += len(payload); files += 1; bytes_out += os.path.getsize(fp)
    os.remove(tmp)
    return {"rows": rows, "files": files, "bytes": bytes_out, "downloaded": downloaded,
            "secs": time.time()-t0, "outdir": outdir}

zip_stats = ingest_zip(ZIP_ENTITY, "http://127.0.0.1:8900/reference.zip")
print(json.dumps({k:v for k,v in zip_stats.items() if k!="outdir"}, indent=2, default=str))
assert zip_stats["rows"] == 450 and zip_stats["files"] == 3
print("PASS: 3 members extracted to gzipped NDJSON, streamed download, flat memory")

{
  "rows": 450,
  "files": 3,
  "bytes": 3506,
  "downloaded": 3837,
  "secs": 0.008841753005981445
}
PASS: 3 members extracted to gzipped NDJSON, streamed download, flat memory


## 6 — CU accounting: make ingestion cost attributable
A Python notebook does not report per-notebook vCore usage to you directly, but the arithmetic is
documented and simple: **CU-seconds = vCore-seconds ÷ 2**. Logging an *estimate* per run gives you
per-entity cost attribution between capacity-metrics refreshes, and a trend line that shows a crawl
degrading long before anyone notices the bill.

In [9]:
def log_run(entity_id, stats, engine, vcores, status="OK", error=None, extra=None):
    dur = float(stats["secs"])
    cu_seconds = dur * vcores / 2.0          # FABRIC_DOC: 1 CU = 2 vCores
    e = extra or {}
    cur.execute("""INSERT INTO etl_run_log(run_id, entity_id, started_at, ended_at, status,
        rows_read, bytes_landed, files_landed, requests_made, throttle_waits, engine, vcores,
        duration_seconds, cu_seconds, error_message)
        VALUES (?,?,?,?,?,?,?,?,?,?,?,?,?,?,?)""",
        (RUN_ID, entity_id, datetime.now(timezone.utc).isoformat(),
         datetime.now(timezone.utc).isoformat(), status, stats.get("rows"), stats.get("bytes"),
         stats.get("files"), e.get("requests"), e.get("waits"), engine, vcores,
         round(dur,3), round(cu_seconds,3), error))
    con.commit()
    return cu_seconds

cu_api = log_run(API_ENTITY, api_stats, "python-notebook", NOTEBOOK_VCORES,
                 extra={"requests": api_stats["requests"], "waits": api_stats["waits"]})
cu_zip = log_run(ZIP_ENTITY, zip_stats, "python-notebook", NOTEBOOK_VCORES)

print(f"{'entity':<16}{'engine':<18}{'secs':>8}{'CU-s':>10}{'rows':>8}{'files':>7}")
for r in cur.execute("""SELECT e.entity_name, l.engine, l.duration_seconds, l.cu_seconds,
                               l.rows_read, l.files_landed
                        FROM etl_run_log l JOIN etl_entity e ON e.entity_id=l.entity_id
                        WHERE l.run_id=?""", (RUN_ID,)):
    print(f"{r[0]:<16}{r[1]:<18}{r[2]:>8.2f}{r[3]:>10.2f}{r[4]:>8}{r[5]:>7}")

# The comparison that justifies the engine choice (Sec 21):
spark_equiv = api_stats["secs"] * 8 / 2      # one Medium node (8 vCores) idling through the same crawl
print(f"\nSame crawl on a Medium Spark node: ~{spark_equiv:.2f} CU-s vs {cu_api:.2f} CU-s "
      f"({spark_equiv/max(cu_api,0.01):.1f}x) - and most of it spent asleep honouring the rate limit.")

entity          engine                secs      CU-s    rows  files
orders_api      python-notebook       1.05      1.05    1000      5
reference_zip   python-notebook       0.01      0.01     450      3

Same crawl on a Medium Spark node: ~4.20 CU-s vs 1.05 CU-s (4.0x) - and most of it spent asleep honouring the rate limit.


## 7 — Cross-workspace rollup: what is actually supported
The Capacity Metrics App is the authority for billable CU, breaking consumption down by item and
operation — but note two constraints before you build on it:

- **Its semantic model is not a supported integration point.** Microsoft's documentation states that
  consumption from, usage of, or modification of the app's semantic model isn't supported, and the
  in-app history is short-lived. Treat it as a *reconciliation* surface, not an ETL source.
- **Not everything is reported.** Library-management consumption and system Spark jobs (including
  those for live pools/sessions) are excluded, and all Spark work is classified as *background*,
  attributed to the notebook, SJD or lakehouse item.

So the durable pattern is the inverse of the brief's suggestion: **your own `etl_run_log` is the
cross-workspace rollup** — every run writes its entity, engine, duration and CU estimate to one
central Fabric SQL Database regardless of which workspace it ran in. Reconcile that estimate against
the Capacity Metrics App periodically by eye (expect drift: your estimate omits session startup,
library install and idle session time), and alert on *your* numbers, which are available in seconds
rather than after a 10–15 minute refresh.

In [10]:
print("--- what a cross-workspace rollup query looks like against your own log ---")
print("""SELECT workspace_name, entity_name, COUNT(*) AS runs,
       ROUND(SUM(cu_seconds)/3600.0, 3) AS cu_hours,
       ROUND(AVG(duration_seconds), 1)  AS avg_secs
FROM   dbo.etl_run_log l JOIN dbo.etl_entity e ON e.entity_id = l.entity_id
WHERE  l.started_at >= DATEADD(day, -7, SYSUTCDATETIME())
GROUP BY workspace_name, entity_name
ORDER BY cu_hours DESC;""")
for s in (srv, zsrv): s.shutdown()
con.close()
print("\nservers stopped, metadata connection closed")

--- what a cross-workspace rollup query looks like against your own log ---
SELECT workspace_name, entity_name, COUNT(*) AS runs,
       ROUND(SUM(cu_seconds)/3600.0, 3) AS cu_hours,
       ROUND(AVG(duration_seconds), 1)  AS avg_secs
FROM   dbo.etl_run_log l JOIN dbo.etl_entity e ON e.entity_id = l.entity_id
WHERE  l.started_at >= DATEADD(day, -7, SYSUTCDATETIME())
GROUP BY workspace_name, entity_name
ORDER BY cu_hours DESC;



servers stopped, metadata connection closed
